# 06 · Evaluation and Packaging

1. **ML results summary** from notebook 02
2. **Retrieval quality** summary from notebooks 03 and 04
3. **13 curated assistant test queries** with **regression-like checks**: route correctness + output assertions
4. The **pytest** regression suite
5. Conclusions, assumptions and limitations

In [ ]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

## 1. Classical ML results

In [ ]:
sup = json.load(open(OUT / "supervised_summary.json"))
print("Supervised:", sup["model"], "| CV ROC-AUC", round(sup["cv_roc_auc"], 3), "| tuned threshold", round(sup["threshold"], 2))
print("Top drivers:", sup["top_drivers"])
display(pd.read_csv(OUT / "02_test_results.csv", index_col=0))
seg = json.load(open(OUT / "segment_summary.json"))
pd.DataFrame(seg["segments"])[["name", "share", "days_since_last_purchase", "total_orders", "avg_order_value", "purchase_rate", "discount_lift"]].round(3)

## 2. Retrieval quality

In [ ]:
display(pd.read_csv(OUT / "03_retrieval_inspection.csv"))
pe = pd.read_csv(OUT / "04_policy_retrieval.csv")
print(f"policy retrieval: hit@1 {pe['hit@1'].mean():.0%}, hit@3 {pe['hit@3'].mean():.0%} on {len(pe)} labelled questions")

## 3. Curated test queries with regression checks

In [ ]:
from src.evaluation import run_suite, CASES
from src.llm import USAGE
res = run_suite(verbose=False)
print(f"PASS {res.PASS.sum()}/{len(res)} ({res.PASS.mean():.0%}) | routing correct {res.route_ok.mean():.0%} | "
      f"avg latency {res.latency_s.mean():.2f}s | LLM tokens {res.llm_tokens.sum()} | API usage {USAGE}")
res[["id", "query", "expected_tools", "routed_tools", "checks_passed", "failed_checks", "PASS", "latency_s"]]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 0.42 * len(res) + 1)); ax.axis("off")
cols = ["id", "query", "routed_tools", "checks_passed", "PASS", "latency_s"]
tb = ax.table(cellText=res[cols].astype(str).replace("₹", "Rs ", regex=True).values, colLabels=cols, loc="center", cellLoc="left", colWidths=[.04, .52, .2, .09, .06, .08])
tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.35)
for (r, c), cell in tb.get_celld().items():
    if r == 0: cell.set_facecolor("#1f3b57"); cell.set_text_props(color="white", weight="bold")
    elif cols[c] == "PASS": cell.set_facecolor("#d3f9d8" if res.iloc[r - 1].PASS else "#ffe3e3")
ax.set_title(f"Assistant test suite: {res.PASS.mean():.0%} pass ({mode_label()})", weight="bold")
plt.tight_layout(); plt.savefig(OUT / "06_test_suite.png", dpi=130); plt.show()

In [ ]:
# full answers for the five example queries from the brief
log = [json.loads(l) for l in open(OUT / "eval_log.jsonl")]
for r in log[:5]:
    print(f"[{r['id']}] {r['query']}\n{r['final_answer']}\n" + "-" * 100)

## 4. pytest regression suite (offline, deterministic)

In [ ]:
import subprocess
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:warnings", str(ROOT / "tests")],
                   capture_output=True, text=True, cwd=ROOT)
print(t.stdout[-1200:])
json.dump({"suite_pass_rate": float(res.PASS.mean()), "suite_pass": int(res.PASS.sum()), "suite_total": len(res),
           "routing_accuracy": float(res.route_ok.mean()), "pytest": t.stdout.strip().splitlines()[-1] if t.stdout else "n/a",
           "mode": mode_label(), "usage": USAGE}, open(OUT / "evaluation_summary.json", "w"), indent=2)

## 5. Conclusions

**What works**
- A clean, leakage-aware ML pipeline. Conversion is driven mainly by **membership tier, income band and product rating**. Price and discount matter little *on average*.
- The RFM segmentation shows **where** discounts work: the *Frequent Small-Basket Buyers* convert about 14 points more on deep discounts, while the other segments barely react. Two complementary views for the business.
- Product recommendations come only from retrieved items, with budget and attributes enforced as hard filters and a validator that removes anything ungrounded.
- Policy answers cite `policy_doc_id`, quote the rule verbatim, and refuse when the policies are silent.
- The controller is simple and fully traceable. It handles multi-intent queries, session memory and guardrails, with at most 3 tool calls per query.

**Assumptions**
- Each interaction row is one catalogue item and one customer snapshot, because the source ids are not stable keys.
- The template-generated text columns and the 5 provided policies are treated as the store's real content. No policy text was invented.
- Prices are in INR, and stock above 0 means the item can be recommended.

**Limitations and next steps**
- The tabular signal is modest (ROC-AUC about 0.65-0.68). Real clickstream data (sessions, previous views) would help most.
- The segments are only moderately separated (silhouette about 0.26).
- The review text is templated (28 variants), so review summaries are generic.
- The image column has no files, so the optional multimodal extension was not attempted.
- Next steps: a reranker for retrieval, an LLM-as-judge groundedness score, and A/B testing segment-specific discount strategies.